# 📦 wheels を落としてオフライン install を測る（施策 B の実証）

3.94 GB / 198 files を Dataset 化した場合の install 所要を実測。


In [ ]:
# 施策 B の実証: wheels を落として **オフライン install** を測る
# GPU 不要。Kaggle は 1 セッション使い捨てなので、torch を外して cold を再現する。
import glob
import json
import os
import subprocess
import sys
import time

SPEC = "vllm==0.32.0"
WD = "/kaggle/working/wheels"
os.makedirs(WD, exist_ok=True)


def run(cmd, label, timeout=5400):
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    dt = time.time() - t0
    print(f"[{label}] {dt:.1f}s rc={r.returncode}")
    if r.returncode != 0:
        print(((r.stdout or "") + (r.stderr or ""))[-1200:])
    return dt, r


res = {}

# 1. 取得（Dataset 化する中身）
dt, _ = run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
             SPEC, "-d", WD], "download")
files = glob.glob(WD + "/*")
res["download_sec"] = dt
res["files"] = len(files)
res["bytes"] = sum(os.path.getsize(f) for f in files)
print(f"  {res['bytes']/1e9:.2f} GB / {res['files']} files")
print(sorted(os.path.basename(f) for f in files)[:15])

# 2. cold 再現のため重い所を外す（このセッションは捨てる前提）
run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch"],
    "uninstall")

# 3. pip オフライン install
dt, r = run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
             "--no-index", "--find-links", WD, SPEC], "pip offline")
res["pip_offline_sec"] = dt
res["pip_offline_ok"] = r.returncode == 0

# 4. uv オフライン install（比較）
run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm"], "uninstall vllm")
run([sys.executable, "-m", "pip", "install", "-q", "--no-input", "uv"],
    "install uv")
dt, r = run(["uv", "pip", "install", "--system", "--no-index",
             "--find-links", WD, "-q", SPEC], "uv offline")
res["uv_offline_sec"] = dt
res["uv_offline_ok"] = r.returncode == 0

# 5. 復元確認
r = subprocess.run([sys.executable, "-c",
                    "import vllm, torch;"
                    "print(vllm.__version__, torch.__version__)"],
                   capture_output=True, text=True)
print("import:", r.returncode, (r.stdout or "").strip(),
      (r.stderr or "")[-300:])
res["import_ok"] = r.returncode == 0

# 成果物に 4 GB を含めない（取得に時間がかかるだけ）。数値だけ残す。
import shutil
shutil.rmtree(WD, ignore_errors=True)

with open("/kaggle/working/wheelset.json", "w") as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print(json.dumps(res, indent=2, ensure_ascii=False))
